# Pyannote Diarization testing - pyannote/pyannote-audio
# <https://github.com/pyannote/pyannote-audio>

## `community-1` open-source speaker diarization

In [1]:
audio_file = "dummy_data/sample_audio.mp3"


In [ ]:
import log_config # to override default and use loguru instead
log_config.setup_logging()
from loguru import logger


1. Make sure [`ffmpeg`](https://ffmpeg.org/) is installed on your machine (needed by [`torchcodec`](https://docs.pytorch.org/torchcodec/) audio decoding library)


In [2]:
# TODO figure out how to ensure ffmpeg is installed... apt-get or other pkgmgr type thing? bundle it locally?


In [3]:
%%bash
ffmpeg


bash: line 1: ffmpeg: command not found


CalledProcessError: Command 'b'ffmpeg\n'' returned non-zero exit status 127.

2. Install with [`uv`](https://docs.astral.sh/uv/)`add pyannote.audio` (recommended) or `pip install pyannote.audio`


In [4]:
%%bash
uv add pyannote.audio


Resolved 357 packages in 6ms
Audited 331 packages in 112ms


3. Accept [`pyannote/speaker-diarization-community-1`](https://hf.co/pyannote/speaker-diarization-community-1) user conditions


Check!

4. Create Huggingface access token at [`hf.co/settings/tokens`](https://hf.co/settings/tokens)


Check! Located in the local `.env` file under name `hf_token`. An example `.env` file can be found at `.env.sample`. Accessible in the code as `settings.hf_token`. Setup for this config is below:

In [5]:
# setup stuff

from pydantic_settings import BaseSettings, SettingsConfigDict
# from pydantic import BaseSettings

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8')

settings = NotebookSettings()

#print(f"HF_TOKEN env variable: {settings.hf_token}")


In [ ]:
# TODO this chunk is yielding:
# c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
# torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
# * use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
# * fix torchcodec installation. Error message was:

# Could not load libtorchcodec. Likely causes:
#           1. FFmpeg is not properly installed in your environment. We support
#              versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
#              for each of those versions. Errors for versions not installed on
#              your system are expected; only the error for your installed FFmpeg
#              version is relevant. On Windows, ensure you've installed the
#              "full-shared" version which ships DLLs.
#           2. The PyTorch version (2.10.0+cpu) is not compatible with
#              this version of TorchCodec. Refer to the version compatibility
#              table:
#              https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
#           3. Another runtime dependency; see exceptions below.
import torch
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

# Community-1 open-source speaker diarization pipeline
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token = settings.hf_token
    )

# send pipeline to GPU (when available)
pipeline.to(torch.device("cuda"))

# apply pretrained pipeline (with optional progress hook)
with ProgressHook() as hook:
    output = pipeline(audio_file, hook = hook)  # runs locally

# print the result
for turn, speaker in output.speaker_diarization:
    print(f"start = {turn.start:.1f}s stop = {turn.end:.1f}s speaker_{speaker}")
# start=0.2s stop=1.5s speaker_0
# start=1.8s stop=3.9s speaker_1
# start=4.2s stop=5.7s speaker_0
# ...


c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibili

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RobynPfeifer\.cache\huggingface\hub\models--pyannote--speaker-diarization-community-1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL pyannote.audio.core.task.Specifications was not an allowed global by default. Please use `torch.serialization.add_safe_globals([pyannote.audio.core.task.Specifications])` or the `torch.serialization.safe_globals([pyannote.audio.core.task.Specifications])` context manager to allowlist this global if you trust this class/function.

Check the documen

In [ ]:
import hf


In [ ]:
%%bash
hf auth login


In [ ]:
# example usage from HF page use model dialog

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization")

# inference on the whole file
pipeline(audio_file) # originally .wav, trying mp3

# inference on an excerpt
from pyannote.core import Segment
excerpt = Segment(start = 2.0, end = 5.0)

from pyannote.audio import Audio
waveform, sample_rate = Audio().crop("file.wav", excerpt)
pipeline({"waveform": waveform, "sample_rate": sample_rate})


In [ ]:
# example usage from HF page itself

# 1. visit hf.co/pyannote/speaker-diarization and accept user conditions
# 2. visit hf.co/pyannote/segmentation and accept user conditions
# 3. visit hf.co/settings/tokens to create an access token
# 4. instantiate pretrained speaker diarization pipeline
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization@2.1",
                                    use_auth_token = settings.hf_token
                                    )


# apply the pipeline to an audio file
diarization = pipeline(audio_file)

# dump the diarization output to disk using RTTM format
with open("audio.rttm", "w") as rttm:
    diarization.write_rttm(rttm)
